# DSSATLab: connect and install

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdelrahmanAmr3/dssatlab/blob/master/notebook/dssatlab_walkthrough.ipynb)

A quick test of the two things `dssatlab` does today, using the latest code from GitHub (before the next PyPI release):

1. `connect()` finds DSSAT.
2. `install()` builds DSSAT on Linux/Colab.

Run the cells top to bottom. The build in step 3 takes several minutes.

## Setup

In [ ]:
%pip install -q "git+https://github.com/AbdelrahmanAmr3/dssatlab.git"

In [ ]:
import logging
import dssatlab as dl

# dssatlab logs how it finds DSSAT; show those messages here.
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
print("dssatlab", dl.__version__)

## 1. `connect()` before installing

On a fresh runtime DSSAT is not installed, so we expect `DSSATNotFoundError`. It lists what was checked and what to do next.
`interactive=False` raises the error instead of asking a question (normally `dl.connect()` would offer to install).

In [ ]:
try:
    print(dl.connect(interactive=False))
except dl.DSSATNotFoundError as e:
    print(e)

## 2. `install()`

Downloads the latest stable DSSAT release from GitHub and builds it with git, cmake and gfortran (already on Colab).

In [ ]:
import time

t0 = time.time()
dssat = dl.install()
print(dssat)
print(f"took {time.time() - t0:.0f}s")

## 3. `connect()` after installing

In [ ]:
# The build was remembered, so connect() now returns it straight away.
assert dl.connect(interactive=False) == dssat

## 4. `install()` again

The build is reused, so this should take a moment instead of minutes.

In [ ]:
t0 = time.time()
again = dl.install()
print(again, f"({time.time() - t0:.1f}s)")
assert again == dssat

## 5. Reading results

Run the sample maize FileX for all six treatments: rainfed, irrigated and vegetative stress, each with low or high nitrogen. We copy the sample folder so the installed files stay untouched.

In a verified sample run, rainfed yield was 2293 kg/ha at both nitrogen levels; irrigated yield was 8207 kg/ha with low nitrogen and 11854 kg/ha with high nitrogen.

For plots, install `pip install dssatlab[plot]`. Both plots below compare treatments from the same run directory. Pandas is optional: install `pip install pandas` to run the final DataFrame cell, or skip it.


In [ ]:
from pathlib import Path
import shutil
import tempfile

sample_dir = dssat.parent / "Maize"
work_dir = Path(tempfile.mkdtemp(prefix="dssatlab_results_"))
maize_dir = work_dir / "Maize"
shutil.copytree(sample_dir, maize_dir)

# Omitting treatment runs all six treatments in this FileX.
result = dl.run(maize_dir / "UFGA8201.MZX", executable=dssat)
print(result.run_dir)


In [ ]:
for row in result.summary():
    print(f"{row['TNAM']}: {row['HWAM']} kg/ha")


In [ ]:
dl.plot_plant_growth(result.run_dir, "LAID")


In [ ]:
result.plot("CWAD")


In [ ]:
dl.to_dataframe(result.summary())
